# DistilBERT vs logistic regression

Logistic regression on TF-IDF reached 0.908 macro F1 on the full dataset. This notebook fine-tunes a small transformer, DistilBERT, on the same task and compares the two on the same training and test reviews.

Run it on Google Colab with a GPU (Runtime > Change runtime type > T4 GPU). First make the sample on your own machine with `python make_transformer_sample.py`, then upload `data/transformer_sample.parquet` when the notebook asks for it.

In [ ]:
import os
import sys
import time

import pandas as pd
import torch
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 1. Load the sample

Same sampling and train/test split as the scikit-learn vs Spark benchmark. `text` is the raw review, `clean_text` is the cleaned version used by logistic regression.

In [ ]:
SAMPLE = "data/transformer_sample.parquet"
if not os.path.exists(SAMPLE):
    SAMPLE = "transformer_sample.parquet"
    if "google.colab" in sys.modules and not os.path.exists(SAMPLE):
        from google.colab import files
        files.upload()

reviews = pd.read_parquet(SAMPLE)
train = reviews[reviews.split == "train"].reset_index(drop=True)
test = reviews[reviews.split == "test"].reset_index(drop=True)
print(f"{len(train):,} training reviews, {len(test):,} test reviews")
print(f"Positive share: {train.label.mean():.1%}")

## 2. Logistic regression on the same sample

The exact model from `train_one.py`: TF-IDF of words and word pairs, hashed into 262,144 slots, balanced class weights.

In [ ]:
start = time.time()
logreg = make_pipeline(
    HashingVectorizer(n_features=2 ** 18, ngram_range=(1, 2), alternate_sign=False, token_pattern=r"\S+"),
    TfidfTransformer(),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)
logreg.fit(train.clean_text, train.label)
logreg_seconds = time.time() - start
logreg_pred = logreg.predict(test.clean_text)

print(f"Macro F1: {f1_score(test.label, logreg_pred, average='macro'):.4f} ({logreg_seconds:.0f} s)")

## 3. Fine-tune DistilBERT

DistilBERT is a smaller version of BERT, already trained on a lot of English text. Fine-tuning adds a classification layer on top and trains the whole network a little more on our reviews.

It reads the raw text, not the cleaned one: it was trained on normal sentences, and it understands word order, so "not good" and "good, not bad" mean different things to it. Reviews are cut at 128 tokens.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 2
LEARNING_RATE = 2e-5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def encode(texts, labels):
    tokens = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH)
    return [{"input_ids": ids, "attention_mask": mask, "labels": int(label)}
            for ids, mask, label in zip(tokens["input_ids"], tokens["attention_mask"], labels)]


# pads each batch to its longest review instead of padding everything to 128
collate = DataCollatorWithPadding(tokenizer)
train_loader = torch.utils.data.DataLoader(encode(train.text, train.label), batch_size=BATCH_SIZE,
                                           shuffle=True, collate_fn=collate)
test_loader = torch.utils.data.DataLoader(encode(test.text, test.label), batch_size=128,
                                          collate_fn=collate)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

# same idea as class_weight="balanced": mistakes on the rarer negative reviews cost more
counts = train.label.value_counts().sort_index()
class_weights = torch.tensor((len(train) / (2 * counts)).values, dtype=torch.float).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
# half precision on the GPU, about twice as fast
scaler = torch.amp.GradScaler(device, enabled=device == "cuda")

start = time.time()
for epoch in range(EPOCHS):
    model.train()
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=device == "cuda"):
            loss = loss_fn(model(**batch).logits, labels)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if step % 500 == 0:
            print(f"epoch {epoch + 1}, step {step}/{len(train_loader)}, loss {loss.item():.3f}")
bert_seconds = time.time() - start
print(f"Training took {bert_seconds / 60:.1f} minutes")

In [ ]:
model.eval()
bert_pred = []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        batch.pop("labels")
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=device == "cuda"):
            bert_pred += model(**batch).logits.argmax(dim=1).tolist()

print(f"Macro F1: {f1_score(test.label, bert_pred, average='macro'):.4f}")

## 4. Results

In [ ]:
def scores(name, predictions, seconds):
    f1_neg, f1_pos = f1_score(test.label, predictions, average=None)
    return {"model": name,
            "macro_f1": round(f1_score(test.label, predictions, average="macro"), 4),
            "f1_negative": round(f1_neg, 4),
            "f1_positive": round(f1_pos, 4),
            "accuracy": round(accuracy_score(test.label, predictions), 4),
            "training_seconds": round(seconds)}


results = pd.DataFrame([
    scores("Logistic regression (TF-IDF)", logreg_pred, logreg_seconds),
    scores("DistilBERT (fine-tuned)", bert_pred, bert_seconds),
])
results

In [ ]:
test["logreg"] = logreg_pred
test["bert"] = bert_pred
fixed = test[(test.logreg != test.label) & (test.bert == test.label)]
broken = test[(test.logreg == test.label) & (test.bert != test.label)]
print(f"DistilBERT is right where logistic regression is wrong: {len(fixed):,} reviews")
print(f"Logistic regression is right where DistilBERT is wrong: {len(broken):,} reviews")


def shorten(text, length=200):
    text = " ".join(text.replace("<br />", " ").split())
    return text if len(text) <= length else text[:length] + "..."


examples = fixed.sample(min(5, len(fixed)), random_state=1)
for _, row in examples.iterrows():
    label = "positive" if row.label == 1 else "negative"
    print(f"- {label}: {shorten(row.text)}")

In [ ]:
gpu = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
lines = [
    "# DistilBERT vs logistic regression",
    "",
    f"Trained on {len(train):,} reviews, tested on {len(test):,} reviews (same sample for both models). "
    f"DistilBERT: {EPOCHS} epochs, reviews cut at {MAX_LENGTH} tokens, on {gpu}.",
    "",
    "| " + " | ".join(results.columns) + " |",
    "|" + "---|" * len(results.columns),
    *["| " + " | ".join(str(v) for v in row) + " |" for row in results.values],
    "",
    f"DistilBERT is right where logistic regression is wrong on {len(fixed):,} test reviews, "
    f"and wrong where it is right on {len(broken):,}.",
    "",
    "Reviews DistilBERT gets right and logistic regression gets wrong:",
    "",
]
for _, row in examples.iterrows():
    label = "positive" if row.label == 1 else "negative"
    lines.append(f'- {label}: "{shorten(row.text)}"')

os.makedirs("results", exist_ok=True)
with open("results/transformer.md", "w") as f:
    f.write("\n".join(lines) + "\n")
print("\n".join(lines))

if "google.colab" in sys.modules:
    from google.colab import files
    files.download("results/transformer.md")